# 📘 Notebook 1.2 — Base Tables Profiling & Structural Data Validation

## 1️⃣ Purpose of This Notebook

This notebook performs a **forensic inspection** of all **base (source) tables**.

**Critical Function**:
> *"Is the raw data structurally sound, physically possible, and legally valid?"*

**Mandate**:
If something is wrong here (e.g., negative revenue), everything later becomes questionable. We must certify the raw materials before building the factory.

---

## 2️⃣ Scope Definition (Strict)

**In Scope Targets**:
1.  **Location_Metadata**: Static geospatial attributes.
2.  **Business_Profile**: Firmographics and operating costs.
3.  **RealTime_Metrics**: Raw high-frequency IoT signals.
4.  **Weather**: Environmental context.
5.  **Events**: Sparse external factors.
6.  **Promotions**: Marketing interventions.
7.  **Daily_Revenue**: The ground truth target variable.

**Out of Scope**:
*   Derived metrics (Hourly/Daily aggregated).
*   Model outputs.
*   Cross-table joins (handled in EDA).

---

In [21]:
import pandas as pd
import numpy as np
import os
import glob

# Define Path to CSV Data (Simulating Raw Ingestion Source)
DATA_DIR = '../../Dataset_Tables/csv_exports/'

def load_table(pattern):
    """Loads the first file matching the pattern from the data directory."""
    files = glob.glob(os.path.join(DATA_DIR, pattern))
    if not files:
        print(f"⚠️ No file found for pattern: {pattern}")
        return None
    file_path = files[0]
    print(f"✅ Loading: {os.path.basename(file_path)}")
    return pd.read_csv(file_path)

print(f"Data Source: {os.path.abspath(DATA_DIR)}")

## 3️⃣ Profiling Framework

We apply these three checks to every table:

### A. Structural Validation (Schema)
*   **Action**: UUID check, Column listing, Type check.

### B. Cardinality & Uniqueness
*   **Action**: Count distinct PKs vs Total Rows.

### C. Value Range & Sanity
*   **Action**: Detect negative prices, zero dwell times, or future dates.

---

In [22]:
def check_structure(df, name, pk_col=None):
    print(f"\n🔍 INSPECTION: {name}")
    print(f"   - Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Cardinality Check
    if pk_col and pk_col in df.columns:
        unique_ids = df[pk_col].nunique()
        total_rows = len(df)
        is_unique = unique_ids == total_rows
        print(f"   - Primary Key ({pk_col}): {'✅ Unique' if is_unique else f'❌ DUPLICATES FOUND ({total_rows - unique_ids})'}")
    else:
        print(f"   - Primary Key: Not specified or not found ({pk_col})")

def check_nulls(df):
    nulls = df.isnull().sum()
    if nulls.sum() == 0:
        print("   - Missing Values: ✅ None")
    else:
        print("   - Missing Values: ⚠️ FOUND")
        print(nulls[nulls > 0])

def check_logic(condition, message):
    """Evaluates a boolean series and reports the count of True values (Violations)."""
    violation_count = condition.sum()
    status = "✅ PASS" if violation_count == 0 else f"❌ FAIL ({violation_count} rows)"
    print(f"   - Logic Check [{message}]: {status}")

## 4️⃣ Validation Dimension 1 — Location Metadata

**Objective**: Validate the physical nodes of the network.
**Checks**:
1.  **Uniqueness**: `location_id` must be unique.
2.  **Geospatial Validity**: Lat/Long not 0,0.
3.  **Catchment Consistency**: Radius > 0.

In [23]:
df_loc = load_table('location_metadata*.csv')

if df_loc is not None:
    check_structure(df_loc, 'Location_Metadata', pk_col='location_id')
    check_nulls(df_loc)
    
    # Logic Check 1: Catchment Radius must be positive
    check_logic(df_loc['catchment_radius_m'] <= 0, "Catchment Radius > 0")
    
    # Logic Check 2: Competitor Count cannot be negative
    check_logic(df_loc['competitor_count'] < 0, "Competitor Count >= 0")
    
    print("\n   - Value Distribution:")
    print(df_loc[['latitude', 'longitude', 'population', 'avg_income']].describe().T[['min', 'max', 'mean']])

## 5️⃣ Validation Dimension 2 — Business Profile

**Objective**: Validate economic characteristics.
**Checks**:
1.  **Keys**: `business_id` uniqueness.
2.  **Cost Reality**: Rent/Costs > 0.
3.  **Operational Validity**: Opening date valid.

In [24]:
df_biz = load_table('business_profile*.csv')

if df_biz is not None:
    check_structure(df_biz, 'Business_Profile', pk_col='business_id')
    check_nulls(df_biz)
    
    # Logic Check: Costs must be positive
    cost_cols = [c for c in df_biz.columns if 'cost' in c or 'rent' in c]
    for col in cost_cols:
        check_logic(df_biz[col] < 0, f"{col} >= 0")

    # Logic Check: Floor area must be realistic
    check_logic(df_biz['store_area_m2'] <= 5, "Store Area > 5 m2")

## 6️⃣ Validation Dimension 3 — RealTime Metrics

**Objective**: Validate high-frequency IoT signals.
**Checks**:
1.  **Physical Possibility**: `people_count` >= 0.
2.  **Time Physics**: `dwell_time_avg` >= 0.
3.  **Congestion**: Valid levels.

In [25]:
df_real = load_table('realtime_metrics*.csv')

if df_real is not None:
    # Note: PK is composite (location_id, timestamp), so simple uniq check on one col won't work well here.
    check_structure(df_real, 'RealTime_Metrics')
    
    # Physics Check 1: People Count
    check_logic(df_real['people_count'] < 0, "People Count >= 0")
    
    # Physics Check 2: Dwell Time
    check_logic(df_real['dwell_time_avg'] < 0, "Dwell Time >= 0")
    
    # Congestion Level Check (Categorical/Ordinal)
    print(f"   - Unique Congestion Levels: {df_real['congestion_level'].unique()}")
    
    # Time Coverage Estimate
    if 'timestamp' in df_real.columns:
        t_min = df_real['timestamp'].min()
        t_max = df_real['timestamp'].max()
        print(f"   - Time Coverage: {t_min} to {t_max}")

## 7️⃣ Validation Dimension 4 — Weather Data

**Objective**: Validate environmental context.
**Checks**:
1.  **Thermal Reality**: Temp -50 to +60 C.
2.  **Hydrological Reality**: Precip >= 0.

In [26]:
df_weather = load_table('weather*.csv')

if df_weather is not None:
    check_structure(df_weather, 'Weather', pk_col='weather_id')
    
    # Thermal Reality Check (Sanity bounds for urban environments)
    check_logic((df_weather['temp_c'] < -50) | (df_weather['temp_c'] > 60), "Temp between -50 and 60 C")
    
    # Rain Check
    check_logic(df_weather['precip_mm'] < 0, "Precipitation >= 0")

## 8️⃣ Validation Dimension 5 — Promotions

**Objective**: Validate marketing interventions.
**Checks**:
1.  **Temporal Logic**: Start Date <= End Date.
2.  **Magnitude**: Lift % reasonable.

In [27]:
df_promo = load_table('promotions*.csv')

if df_promo is not None:
    check_structure(df_promo, 'Promotions', pk_col='promo_id')
    
    # Temporal Logic Check
    # Ensure we are comparing dates, simple string comparison works for ISO format YYYY-MM-DD
    # Fix applied: passing valid Boolean Mask Series instead of Scalar boolean
    check_logic(df_promo['start_date'] > df_promo['end_date'], "Start Date <= End Date")
    
    # Lift Reality
    print("   - Lift Distribution:")
    print(df_promo['expected_lift_pct'].describe().T[['min', 'max', 'mean']])

## 9️⃣ Validation Dimension 6 — Daily Revenue (Target)

**Objective**: Validate the ground truth metric.
**Checks**:
1.  **Non-Negativity**: Revenue >= 0.
2.  **Transactions**: Trans >= 0.
3.  **Consistency**: Rev > 0 implies Trans > 0.

In [28]:
df_rev = load_table('daily_revenue*.csv')

if df_rev is not None:
    # PK is (business_id, date)
    check_structure(df_rev, 'Daily_Revenue')
    check_nulls(df_rev)
    
    # Finance Check 1: Revenue >= 0
    check_logic(df_rev['total_revenue'] < 0, "Revenue >= 0")
    
    # Finance Check 2: Transactions >= 0
    check_logic(df_rev['total_transactions'] < 0, "Transactions >= 0")

    # Transactions vs Revenue Consistency
    # If revenue > 0, transactions should be > 0
    # Fix applied: passing valid Boolean Mask Series
    check_logic((df_rev['total_revenue'] > 0) & (df_rev['total_transactions'] == 0), "Non-zero revenue implies non-zero transactions")

## 🔟 Final Consolidated Cleaning Decisions

**Data Cleaning Contract Candidates**:

| Table | Constraint ID | Detected Issue | Decision (Action) |
| :--- | :--- | :--- | :--- |
| **Location** | `LOC_01` | Invalid Radius (<=0) | **DROP** row |
| **RealTime** | `RTM_01` | Negative People Count | **CLIP** to 0 |
| **RealTime** | `RTM_02` | Negative Dwell Time | **CLIP** to 0 |
| **Promo** | `PRM_01` | Start > End Date | **SWAP** dates or DROP |

**End of Notebook 1.2**
Proceed to Notebook 1.3 for time-series validation.